In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

os.makedirs("big_data_parts", exist_ok=True)

X, y = make_classification(
    n_samples=200_000,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

columns = [f"feature_{i}" for i in range(1, 11)]
df = pd.DataFrame(X, columns=columns)
df["target"] = y

print(df.shape)
df.head()

(200000, 11)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target
0,-0.528484,0.164206,2.214925,0.285769,-1.189389,0.659787,-2.213390,-1.665073,1.749432,1.507525,0
1,-1.590390,0.284698,-0.589027,-5.241153,0.815853,0.182781,-1.868961,0.081353,4.025677,-1.610721,0
2,0.991974,-0.096971,1.320011,0.630263,2.293518,-0.099596,0.169749,1.092092,-0.225622,0.689678,1
3,0.251844,-0.371644,-2.594009,1.005384,-0.854598,-1.419766,-0.364799,1.022312,-0.433713,-2.181322,1
4,-1.321236,-0.807209,-4.147105,1.195384,-3.297078,-1.059710,-0.480132,1.643833,-0.661296,-3.772295,1


In [2]:

parts = [df.iloc[idx] for idx in np.array_split(np.arange(len(df)), 4)]

for i, part in enumerate(parts, start=1):
    part.to_csv(f"big_data_parts/part_{i}.csv", index=False)

print("Создано файлов:", len(os.listdir("big_data_parts")))

Создано файлов: 4


In [3]:
import dask.dataframe as dd

ddf = dd.read_csv("big_data_parts/part_*.csv")

print("Количество partitions:", ddf.npartitions)
print("Тип объекта:", type(ddf))
ddf.head()

Количество partitions: 4
Тип объекта: <class 'dask.dataframe.dask_expr._collection.DataFrame'>


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target
0,-0.528484,0.164206,2.214925,0.285769,-1.189389,0.659787,-2.213390,-1.665073,1.749432,1.507525,0
1,-1.590390,0.284698,-0.589027,-5.241153,0.815853,0.182781,-1.868961,0.081353,4.025677,-1.610721,0
2,0.991974,-0.096971,1.320011,0.630263,2.293518,-0.099596,0.169749,1.092092,-0.225622,0.689678,1
3,0.251844,-0.371644,-2.594009,1.005384,-0.854598,-1.419766,-0.364799,1.022312,-0.433713,-2.181322,1
4,-1.321236,-0.807209,-4.147105,1.195384,-3.297078,-1.059710,-0.480132,1.643833,-0.661296,-3.772295,1


In [4]:
mean_task = ddf["feature_1"].mean()
print(mean_task)

mean_value = mean_task.compute()
print("Среднее feature_1:", mean_value)

<dask_expr.expr.Scalar: expr=ArrowStringConversion(frame=FromMapProjectable(8da2a74))['feature_1'].mean(), dtype=float64>
Среднее feature_1: -0.0007254807396214997


In [5]:
class_stats = ddf.groupby("target")[[
    "feature_1", "feature_2", "feature_3"
]].mean().compute()

print(class_stats)

        feature_1  feature_2  feature_3
target                                 
0       -0.001915   0.992216   0.019433
1        0.000463   0.001114  -0.335145


## 7. Фильтрация большого набора

In [6]:
filtered = ddf[
    (ddf["feature_1"] > 0) &
    (ddf["feature_2"] < 1)
]

print("Количество строк после фильтрации:", len(filtered))

Количество строк после фильтрации: 60264


In [14]:
from dask.distributed import Client

client = Client(processes=False)
print(client)
client

<Client: 'inproc://10.82.119.215/29971/9' processes=1 threads=10, memory=16.00 GiB>


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.82.119.215:8787/status,
Dashboard: http://10.82.119.215:8787/status,Workers: 1
Total threads: 10,Total memory: 16.00 GiB
Status: running,Using processes: False
Comm: inproc://10.82.119.215/29971/9,Workers: 0
Dashboard: http://10.82.119.215:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.82.119.215/29971/12,Total threads: 10
Dashboard: http://10.82.119.215:64819/status,Memory: 16.00 GiB
Nanny: None,


In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataLab11") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.3.0


In [16]:
spark_df = spark.read.csv(
    "big_data_parts/part_*.csv",
    header=True,
    inferSchema=True
)

print("Количество строк:", spark_df.count())
print("Количество partitions:", spark_df.rdd.getNumPartitions())
spark_df.show(5)

Количество строк: 200000
Количество partitions: 8
+-------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+-------------------+--------------------+-------------------+------+
|          feature_1|           feature_2|          feature_3|          feature_4|          feature_5|           feature_6|           feature_7|          feature_8|           feature_9|         feature_10|target|
+-------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+-------------------+--------------------+-------------------+------+
|-0.5284839673794116| 0.16420636919638376| 2.2149251235883143|0.28576885925651996|-1.1893893689838255|  0.6597866625194193| -2.2133900626795793|  -1.66507293659712|   1.749431511998968| 1.5075246244925888|     0|
|-1.5903900395368513|   0.284697947415289|-0.5890267181079856|  -5.24115309423028| 0.8158527634448

In [17]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [f"feature_{i}" for i in range(1, 11)]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

prepared = assembler.transform(spark_df).select("features", "target")
prepared.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|features                                                                                                                                                                                               |target|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|[-0.5284839673794116,0.16420636919638376,2.2149251235883143,0.28576885925651996,-1.1893893689838255,0.6597866625194193,-2.2133900626795793,-1.66507293659712,1.749431511998968,1.5075246244925888]     |0     |
|[-1.5903900395368513,0.284697947415289,-0.5890267181079856,-5.24115309423028,0.8158527634448813,0.18278054841184754,-1.8689610688737974,0.08135324397879029,4.02567

In [18]:
train_df, test_df = prepared.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 160271
Test: 39729


In [19]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target",
    maxIter=50
)

spark_model = lr.fit(train_df)
predictions = spark_model.transform(test_df)

predictions.select(
    "target", "prediction", "probability"
).show(10, truncate=False)

+------+----------+-----------------------------------------+
|target|prediction|probability                              |
+------+----------+-----------------------------------------+
|0     |0.0       |[0.8826168291587594,0.11738317084124061] |
|1     |0.0       |[0.675819340711227,0.32418065928877304]  |
|0     |0.0       |[0.8660463806655346,0.13395361933446537] |
|1     |1.0       |[0.3530714733255305,0.6469285266744695]  |
|1     |1.0       |[0.005440819194528887,0.9945591808054711]|
|0     |0.0       |[0.9311119831656545,0.06888801683434553] |
|0     |0.0       |[0.9450493373283595,0.054950662671640504]|
|0     |0.0       |[0.9715793568711464,0.028420643128853618]|
|1     |1.0       |[0.13263500269468576,0.8673649973053142] |
|0     |0.0       |[0.9454227308691402,0.05457726913085981] |
+------+----------+-----------------------------------------+
only showing top 10 rows


In [20]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Spark MLlib Accuracy:", accuracy)

Spark MLlib Accuracy: 0.8542626293135996


In [13]:
client.close()
spark.stop()